In [1]:
import polars as pl
import torch

from libs.DataLoader import Loader
from libs.RNNModel import RNNModel
from libs.Solver import Solver
from libs.Trainer import Trainer
from libs.TransformerModel import TransformerModel
from libs.WeightedAvgModel import WeightedAvgModel
from libs.constants import *
from libs.utils import NDCG, count_polars, count_params

In [2]:
EMB_DIM = 64
loader = Loader('ur0.01_ir0.01', content_embedding_size=EMB_DIM, batch_size=500_000)
(train_df, val_df), users_df, items_df = loader.load_data(convert_to_pandas=False)

Load metadata
Create lazy interaction datasets
Get unique users/items


  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

Filter embeddings
Process metadata
Compute aggregates


  0%|          | 0/8 [00:00<?, ?it/s]

Filter interactions
Finalize interactions
Count
Train: 62_578 Val: 12_099 Users: 2_309 Items: 120_415


In [3]:
train_df: pl.LazyFrame = train_df.filter(pl.col(TARGET) > 0)  # Interaction: positive
test_df: pl.LazyFrame = val_df.filter(pl.col(TARGET) > 0)  # Interaction: positive

test = test_df.select(ITEM).unique(ITEM)
cold_test = test_df.join(train_df.select(ITEM).unique(), on=ITEM, how="anti")

print(f"Train: {count_polars(train_df):_} Test: {count_polars(test_df):_}")
print(f"Test items: {count_polars(test)} Test cold items: {count_polars(cold_test)}")

Train: 62_457 Test: 1_970
Test items: 1090 Test cold items: 405


In [4]:
model = TransformerModel(
    in_dim=EMB_DIM,
    hidden_dim=128,
    num_layers=3,
    nheads=8,
    dim_feedforward=256,
    dropout=0.3,
    max_seq_len=100,
)

# model = RNNModel(
#     in_dim=EMB_DIM,
#     hidden_dim=128,
#     layer='lstm',
#     num_layers=2,
#     dropout=0.5,
# )

# model = WeightedAvgModel('add', 'linear', alpha=0.5)

print(f"Params: {count_params(model):_}")

Params: 426_624


In [5]:
trainer = Trainer(
    model=model,
    train_interactions=train_df,
    predict_items=test,
    items_metadata=items_df,
    val_ratio=0.3,
    loss_type='cos',
    # loss_margin=0.1,
    # negative_ratio=2,
    num_recent_videos=100,
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
)  # TODO MarginRankingLoss
solver = Solver(
    trainer=trainer,
    candidates_to_keep=500,
    top_per_item=100,
    max_per_user=100,
)

Temporal train/val split with ratio val_ratio 0.3
Count
All: 2309 Train: 2302 Val: 2203 Test: 1090


In [6]:
trainer.fit(epochs=50, users_batch_size=256, verbose=False)

  0%|          | 0/50 [00:00<?, ?it/s]


KeyboardInterrupt



In [7]:
trainer.model.eval()
solver.collect_candidates(users_batch_size=512, items_batch_size=1000)

data:   0%|          | 0/5 [00:00<?, ?it/s]

predict:   0%|          | 0/2 [00:00<?, ?it/s]

In [8]:
results = solver.solve()

predict = pl.DataFrame({
    ITEM: list(results.keys()),
    USER: [[u for u, _ in users_scores] for users_scores in results.values()]
})

In [9]:
for true, true_name in [
    (test_df, "all"),
    (test_df.join(cold_test, on=ITEM, how='inner'), "cold")
]:
    pl.LazyFrame().group_by().agg()
    print(
        f"Test: {true_name} NDCG: {NDCG(predict.to_pandas(), true.group_by(ITEM).agg(pl.col(USER)).collect().to_pandas()):.4f}")

Test: all NDCG: 0.0500
Test: cold NDCG: 0.0671
